In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("qwen/qwen3.8-27b", model_provider="groq")

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9'}}, client=<groq.resources.chat.completions.Completions object at 0x0000010E1F3B47A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000010E1F032FF0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Mov

In [4]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception** is a 2010 science fiction action-thriller film written and directed by **Christopher Nolan**. It is widely regarded as one of the most complex and influential films of the 21st century, known for its intricate plot, mind-bending concepts, and groundbreaking visual effects.\n\n### **Basic Information**\n- **Director**: Christopher Nolan\n- **Release Date**: July 16, 2010 (US)\n- **Runtime**: 148 minutes\n- **Studio**: Warner Bros. Pictures\n- **Music**: Hans Zimmer\n- **Tagline**: *"Your mind is the scene of the crime."*\n\n### **Main Cast**\n- **Leonardo DiCaprio** as **Dom Cobb**\n- **Joseph Gordon-Levitt** as **Arthur**\n- **Elliot Page** as **Ariadne**\n- **Tom Hardy** as **Eames**\n- **Ken Watanabe** as **Saito**\n- **Marion Cotillard** as **Mal Cobb**\n- **Cillian Murphy** as **Robert Fischer**\n- **Tom Berenger** as **Miles Fischer**\n- **Michael Caine** as **Miles**\n\n### **Plot Summary**\nDom Cobb (DiCaprio) is a skilled thief with a specialize

In [7]:
response=model_with_structure.invoke("Provide details about the moview Inception")


In [8]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qxc7t0xvv', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 355, 'total_tokens': 423, 'completion_time': 0.182047936, 'completion_tokens_details': None, 'prompt_time': 0.027919996, 'prompt_tokens_details': None, 'queue_time': 0.122894453, 'total_time': 0.209967932}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c000-92cf-7285-9a03-a54a5ed79aed-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'qxc7t0xvv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 68, 'total

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}